# Step 3: Machine Learning Classifier

Steps 1 and 2 are non-parametric: they look things up. Step 3 is different — we train a real model that learns to map literals to codes from the training data.

**Why we need this:** Steps 1 and 2 work well for literals that look like something we have seen before or have an obvious description match in the reference. But many literals require generalization — learning that words like 'parto', 'cesárea', 'embarazo' tend to predict O-chapter codes, or that 'fractura', 'luxación', 'contusión' map to S/T-chapter codes. A classifier learns these patterns from training.

**The approach:**
- Vectorize literals with TF-IDF (character and word n-grams combined)
- Train Logistic Regression separately for diagnosis codes and procedure codes (they're different coding systems)
- Only train on codes that appear at least 3 times (otherwise the model has nothing to learn)
- Use the classifier's probability score as a confidence signal
- Combine with Step 1 + Step 2 using a confidence-based cascade

Files to upload before running:
- `train_preprocessed.csv` (from Step 0)
- `test_preprocessed.csv` (from Step 0)
- `step2_predictions.csv` (from Step 2)

Files produced at the end:
- `final_predictions.csv` (full detail with all methods)
- `kaggle_submission.csv` (clean two-column file ready for Kaggle)

## 1. Imports and loading

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('train_preprocessed.csv')
test = pd.read_csv('test_preprocessed.csv')
step2 = pd.read_csv('step2_predictions.csv')

# Defensive: ensure no NaN in the text columns
train['Literal_norm'] = train['Literal_norm'].fillna('')
test['Literal_norm'] = test['Literal_norm'].fillna('')

print(f'Training: {len(train):,} rows')
print(f'Test:     {len(test):,} rows')
print(f'Step 2 predictions: {len(step2):,} rows')
print(f'\nStep 2 method breakdown:')
print(step2['final_method'].value_counts())

Training: 13,700 rows
Test:     6,667 rows
Step 2 predictions: 6,667 rows

Step 2 method breakdown:
final_method
exact        3812
fuzzy        2819
retrieval      36
Name: count, dtype: int64


## 2. Split training into diagnosis and procedure

Diagnosis codes (starting with a letter) and procedure codes (starting with a digit) are two different coding systems with different vocabularies. A single classifier trying to predict both would have to learn that 'parto' goes with O-codes for diagnoses and with 10D-codes for procedures, which makes the problem unnecessarily hard.

We train two separate classifiers and route each test literal to the appropriate one.

In [3]:
train_diag = train[train['code_type'] == 'diagnosis'].copy()
train_proc = train[train['code_type'] == 'procedure'].copy()

print(f'Diagnosis training rows: {len(train_diag):,}')
print(f'  Unique codes:          {train_diag.Code.nunique():,}')
print(f'Procedure training rows: {len(train_proc):,}')
print(f'  Unique codes:          {train_proc.Code.nunique():,}')

Diagnosis training rows: 8,753
  Unique codes:          2,301
Procedure training rows: 4,947
  Unique codes:          1,758


## 3. Filter out codes with too few training examples

The long-tail problem: many codes appear only once or twice. A classifier trained on them will almost certainly overfit or guess wrong. Instead, we restrict training to codes with at least `MIN_EXAMPLES` occurrences. The rare codes are left for Steps 1 and 2 to handle.

In [4]:
MIN_EXAMPLES = 3

def filter_by_min(df, min_count):
    counts = df['Code'].value_counts()
    keep_codes = counts[counts >= min_count].index
    return df[df['Code'].isin(keep_codes)].copy()

train_diag_f = filter_by_min(train_diag, MIN_EXAMPLES)
train_proc_f = filter_by_min(train_proc, MIN_EXAMPLES)

print(f'After filtering codes with < {MIN_EXAMPLES} examples:')
print(f'  Diagnosis: {len(train_diag_f):,} rows, {train_diag_f.Code.nunique():,} unique codes')
print(f'  Procedure: {len(train_proc_f):,} rows, {train_proc_f.Code.nunique():,} unique codes')
print(f'\nWe dropped {len(train_diag) - len(train_diag_f):,} diagnosis rows and '
      f'{len(train_proc) - len(train_proc_f):,} procedure rows.')
print('Those rare codes will be handled by Step 1 or Step 2 retrieval.')

After filtering codes with < 3 examples:
  Diagnosis: 6,919 rows, 859 unique codes
  Procedure: 3,366 rows, 535 unique codes

We dropped 1,834 diagnosis rows and 1,581 procedure rows.
Those rare codes will be handled by Step 1 or Step 2 retrieval.


## 4. Define the classifier pipeline

A `FeatureUnion` combines two vectorizers in parallel:
- Character n-grams (3-5 chars) catch typos and morphological variations
- Word n-grams (1-2 words) catch standard medical terminology

The combined features feed into Logistic Regression with L2 regularization. We use `saga` solver because it scales well to many classes and `class_weight='balanced'` so the classifier doesn't ignore rare codes.

This is essentially the FlatSVM / TF-IDF baseline from the survey paper (Perotte et al. 2013), with Logistic Regression instead of SVM.

In [5]:
def make_pipeline():
    """Return a fresh classifier pipeline. We need a new one per code_type."""
    return Pipeline([
        ('features', FeatureUnion([
            ('char', TfidfVectorizer(
                analyzer='char_wb',
                ngram_range=(3, 5),
                min_df=2,
                sublinear_tf=True,
            )),
            ('word', TfidfVectorizer(
                analyzer='word',
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            )),
        ])),
        ('clf', LogisticRegression(
            max_iter=2000,
            solver='saga',
            C=1.0,
            class_weight='balanced',
            random_state=42,
        ))
    ])

## 5. Train the diagnosis classifier

Training time depends on the number of classes (around 1500 diagnosis codes after filtering). Expect 2-4 minutes.

In [6]:
print('Training diagnosis classifier...')
start = time.time()

clf_diag = make_pipeline()
clf_diag.fit(train_diag_f['Literal_norm'].values, train_diag_f['Code'].values)

print(f'Trained in {time.time()-start:.1f}s')
print(f'Number of classes learned: {len(clf_diag.named_steps["clf"].classes_):,}')

Training diagnosis classifier...
Trained in 130.3s
Number of classes learned: 859


## 6. Train the procedure classifier

In [7]:
print('Training procedure classifier...')
start = time.time()

clf_proc = make_pipeline()
clf_proc.fit(train_proc_f['Literal_norm'].values, train_proc_f['Code'].values)

print(f'Trained in {time.time()-start:.1f}s')
print(f'Number of classes learned: {len(clf_proc.named_steps["clf"].classes_):,}')

Training procedure classifier...
Trained in 11.9s
Number of classes learned: 535


## 7. Train a diagnosis-vs-procedure router

At test time we don't know whether each literal should be routed to the diagnosis or procedure classifier. We train a small binary classifier on the training data (where we do know) to predict this from the literal alone.

In [8]:
print('Training diagnosis-vs-procedure router...')
start = time.time()

router = make_pipeline()
router.fit(train['Literal_norm'].values, train['code_type'].values)

print(f'Trained in {time.time()-start:.1f}s')

# Quick check of router on training data (just to sanity-check, not a real evaluation)
router_pred = router.predict(train['Literal_norm'].values)
router_acc = (router_pred == train['code_type']).mean()
print(f'Router accuracy on training (overfit, just a sanity check): {router_acc*100:.1f}%')

Training diagnosis-vs-procedure router...
Trained in 0.8s
Router accuracy on training (overfit, just a sanity check): 84.2%


## 8. Apply classifiers to the test set

For each test literal:
1. The router predicts whether it is diagnosis or procedure
2. The corresponding classifier predicts the actual code
3. We store the predicted code AND the probability of that prediction (confidence)

In [9]:
print('Routing test literals to diagnosis or procedure classifier...')
test['router_pred'] = router.predict(test['Literal_norm'].values)
print(test['router_pred'].value_counts())

Routing test literals to diagnosis or procedure classifier...
router_pred
diagnosis    4033
procedure    2634
Name: count, dtype: int64


In [10]:
print('Running diagnosis classifier on routed literals...')
diag_idx = test[test['router_pred'] == 'diagnosis'].index
diag_texts = test.loc[diag_idx, 'Literal_norm'].values

diag_probs = clf_diag.predict_proba(diag_texts)
best_diag = diag_probs.argmax(axis=1)
diag_codes = clf_diag.named_steps['clf'].classes_[best_diag]
diag_confidence = diag_probs.max(axis=1)

test['clf_code'] = ''
test['clf_confidence'] = 0.0
test.loc[diag_idx, 'clf_code'] = diag_codes
test.loc[diag_idx, 'clf_confidence'] = diag_confidence

print(f'  {len(diag_idx):,} predictions made')
print(f'  Confidence range: {diag_confidence.min():.3f} to {diag_confidence.max():.3f}')

Running diagnosis classifier on routed literals...
  4,033 predictions made
  Confidence range: 0.002 to 0.620


In [11]:
print('Running procedure classifier on routed literals...')
proc_idx = test[test['router_pred'] == 'procedure'].index
proc_texts = test.loc[proc_idx, 'Literal_norm'].values

proc_probs = clf_proc.predict_proba(proc_texts)
best_proc = proc_probs.argmax(axis=1)
proc_codes = clf_proc.named_steps['clf'].classes_[best_proc]
proc_confidence = proc_probs.max(axis=1)

test.loc[proc_idx, 'clf_code'] = proc_codes
test.loc[proc_idx, 'clf_confidence'] = proc_confidence

print(f'  {len(proc_idx):,} predictions made')
print(f'  Confidence range: {proc_confidence.min():.3f} to {proc_confidence.max():.3f}')

Running procedure classifier on routed literals...
  2,634 predictions made
  Confidence range: 0.003 to 0.511


In [12]:
# Some sample classifier predictions
test[['Literal', 'router_pred', 'clf_code', 'clf_confidence']].sample(10, random_state=3)

,Literal,router_pred,clf_code,clf_confidence
1185,TEGD,diagnosis,F840,0.005468
3809,cardiomegalia,diagnosis,B246ZZZ,0.004599
807,Anèmia ferropènica,procedure,2809,0.262403
2981,HBP,diagnosis,N400,0.019138
1749,trasplante de córnea,diagnosis,E8780,0.077796
4989,Desgarro IIº,diagnosis,O701,0.082161
696,TUMORECTOMÍA mama izq,procedure,0HBU0ZZ,0.126432
5282,Rubéola: NO Inmune,diagnosis,Z228,0.270070
2546,IMC 37,diagnosis,Z6843,0.086341
3011,trastorno ansioso depresivo,diagnosis,F418,0.204933


## 9. Validate all three steps on the same 80/20 split

Now we can build the final comparison: Step 1 vs Step 2 vs Step 3 vs Combined. This is the headline result for the presentation.

In [13]:
# Same split as previous steps for fair comparison
train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
val_split = val_split.copy()
val_split['Literal_norm'] = val_split['Literal_norm'].fillna('')

print(f'Validation set: {len(val_split):,} literals')

Validation set: 2,740 literals


In [14]:
# Step 1 on validation (exact + fuzzy with threshold 60)
from rapidfuzz import fuzz, process

s1_lookup = {}
for lit_norm, group in train_split.groupby('Literal_norm'):
    s1_lookup[lit_norm] = group['Code'].value_counts().index[0]

val_split['s1_exact'] = val_split['Literal_norm'].map(s1_lookup)

lookup_keys = list(s1_lookup.keys())
val_unmatched = val_split[val_split['s1_exact'].isna()]

print(f'Running fuzzy matching on {len(val_unmatched):,} validation literals (Step 1 part)...')
fuzzy_codes = {}
for idx, row in val_unmatched.iterrows():
    r = process.extractOne(row['Literal_norm'], lookup_keys, scorer=fuzz.ratio, score_cutoff=60)
    if r is not None:
        fuzzy_codes[idx] = s1_lookup[r[0]]

val_split['s1_code'] = val_split['s1_exact'].copy()
for idx, code in fuzzy_codes.items():
    val_split.loc[idx, 's1_code'] = code
val_split['s1_correct'] = (val_split['s1_code'] == val_split['Code'])

Running fuzzy matching on 1,266 validation literals (Step 1 part)...


In [15]:
# Step 3 (classifier) on validation
# Train new classifiers using only train_split (so the validation evaluation is honest)
print('Training validation classifiers on the train_split only...')
start = time.time()

split_diag = filter_by_min(train_split[train_split.code_type == 'diagnosis'], MIN_EXAMPLES)
split_proc = filter_by_min(train_split[train_split.code_type == 'procedure'], MIN_EXAMPLES)

val_clf_diag = make_pipeline()
val_clf_diag.fit(split_diag['Literal_norm'].values, split_diag['Code'].values)

val_clf_proc = make_pipeline()
val_clf_proc.fit(split_proc['Literal_norm'].values, split_proc['Code'].values)

val_router = make_pipeline()
val_router.fit(train_split['Literal_norm'].values, train_split['code_type'].values)

print(f'Trained validation classifiers in {time.time()-start:.1f}s')

Training validation classifiers on the train_split only...
Trained validation classifiers in 63.4s


In [16]:
# Apply validation classifiers to the val_split
val_split['router_pred'] = val_router.predict(val_split['Literal_norm'].values)

vdiag_idx = val_split[val_split.router_pred == 'diagnosis'].index
vproc_idx = val_split[val_split.router_pred == 'procedure'].index

val_split['clf_code'] = ''
val_split['clf_confidence'] = 0.0

vd_probs = val_clf_diag.predict_proba(val_split.loc[vdiag_idx, 'Literal_norm'].values)
val_split.loc[vdiag_idx, 'clf_code'] = val_clf_diag.named_steps['clf'].classes_[vd_probs.argmax(axis=1)]
val_split.loc[vdiag_idx, 'clf_confidence'] = vd_probs.max(axis=1)

vp_probs = val_clf_proc.predict_proba(val_split.loc[vproc_idx, 'Literal_norm'].values)
val_split.loc[vproc_idx, 'clf_code'] = val_clf_proc.named_steps['clf'].classes_[vp_probs.argmax(axis=1)]
val_split.loc[vproc_idx, 'clf_confidence'] = vp_probs.max(axis=1)

val_split['s3_correct'] = (val_split['clf_code'] == val_split['Code'])
print(f'Step 3 alone accuracy: {val_split.s3_correct.mean()*100:.1f}%')

Step 3 alone accuracy: 27.2%


In [17]:
# Now combine all three for the validation comparison.
# Strategy: Step 1 first. If unresolved AND retrieval has high score, use retrieval.
# Otherwise use classifier prediction.

# We need Step 2 retrieval predictions for the validation set too.
# Since this notebook already has Step 2 results for the real test set,
# we re-run a small retrieval for validation here.
from sklearn.feature_extraction.text import TfidfVectorizer as TV
from sklearn.metrics.pairwise import cosine_similarity

reference = pd.read_csv('reference_preprocessed.csv')
reference['Description_norm'] = reference['Description_norm'].fillna('')

print('Building retrieval index for validation comparison...')
start = time.time()
rc_vec = TV(analyzer='char_wb', ngram_range=(3,5), min_df=2, max_df=0.95, sublinear_tf=True)
rc_ref = rc_vec.fit_transform(reference['Description_norm'])
rw_vec = TV(analyzer='word', ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)
rw_ref = rw_vec.fit_transform(reference['Description_norm'])
print(f'Built in {time.time()-start:.1f}s')

# Retrieve for the validation set
print('Retrieval on validation set...')
v_char = rc_vec.transform(val_split['Literal_norm'])
v_word = rw_vec.transform(val_split['Literal_norm'])

BATCH = 100
ret_codes, ret_scores = [], []
for i in range(0, len(val_split), BATCH):
    end = min(i+BATCH, len(val_split))
    cs = cosine_similarity(v_char[i:end], rc_ref)
    ws = cosine_similarity(v_word[i:end], rw_ref)
    combined = (cs + ws) / 2
    best = combined.argmax(axis=1)
    for j in range(end - i):
        bi = best[j]
        ret_codes.append(reference.iloc[bi]['Code'])
        ret_scores.append(float(combined[j, bi]))
    del cs, ws, combined

val_split['s2_code'] = ret_codes
val_split['s2_score'] = ret_scores
val_split['s2_correct'] = (val_split['s2_code'] == val_split['Code'])
print(f'Step 2 alone accuracy: {val_split.s2_correct.mean()*100:.1f}%')

Building retrieval index for validation comparison...
Built in 20.6s
Retrieval on validation set...
Step 2 alone accuracy: 10.7%


In [18]:
# Combine all three with the cascade logic:
# - Use Step 1 if it produced a prediction
# - Else use Step 2 if its score is reasonably high (>= 0.5)
# - Else fall back to Step 3 classifier

RETRIEVAL_CONFIDENCE_THRESHOLD = 0.5

val_split['combined_code'] = val_split['s1_code']

# Where Step 1 failed AND retrieval score is high, use retrieval
use_retrieval = val_split['combined_code'].isna() & (val_split['s2_score'] >= RETRIEVAL_CONFIDENCE_THRESHOLD)
val_split.loc[use_retrieval, 'combined_code'] = val_split.loc[use_retrieval, 's2_code']

# Where still unresolved, use classifier
use_clf = val_split['combined_code'].isna()
val_split.loc[use_clf, 'combined_code'] = val_split.loc[use_clf, 'clf_code']

val_split['combined_correct'] = (val_split['combined_code'] == val_split['Code'])

In [19]:
# Final headline comparison
n = len(val_split)
print('Validation accuracy comparison (this is the headline for the presentation):')
print(f'{"":40s} {"correct":>10s} {"accuracy":>10s}')
print('-' * 65)
print(f'{"Step 1 only (exact + fuzzy match)":40s} {val_split.s1_correct.sum():>10,} {val_split.s1_correct.mean()*100:>9.1f}%')
print(f'{"Step 2 only (retrieval)":40s} {val_split.s2_correct.sum():>10,} {val_split.s2_correct.mean()*100:>9.1f}%')
print(f'{"Step 3 only (classifier)":40s} {val_split.s3_correct.sum():>10,} {val_split.s3_correct.mean()*100:>9.1f}%')
print(f'{"Combined cascade (Step 1+2+3)":40s} {val_split.combined_correct.sum():>10,} {val_split.combined_correct.mean()*100:>9.1f}%')

Validation accuracy comparison (this is the headline for the presentation):
                                            correct   accuracy
-----------------------------------------------------------------
Step 1 only (exact + fuzzy match)             1,040      38.0%
Step 2 only (retrieval)                         294      10.7%
Step 3 only (classifier)                        746      27.2%
Combined cascade (Step 1+2+3)                 1,044      38.1%


In [20]:
# How complementary are the three steps?
s1f = val_split.s1_correct.fillna(False)
s2f = val_split.s2_correct.fillna(False)
s3f = val_split.s3_correct.fillna(False)

all_right = (s1f & s2f & s3f).sum()
only_s1 = (s1f & ~s2f & ~s3f).sum()
only_s2 = (~s1f & s2f & ~s3f).sum()
only_s3 = (~s1f & ~s2f & s3f).sum()
all_wrong = (~s1f & ~s2f & ~s3f).sum()
any_right = (s1f | s2f | s3f).sum()

print('Where each step uniquely contributes (validation set):')
print(f'  All three correct:    {all_right:>5,}')
print(f'  Only Step 1 correct:  {only_s1:>5,}  (would be missed without matching)')
print(f'  Only Step 2 correct:  {only_s2:>5,}  (would be missed without retrieval)')
print(f'  Only Step 3 correct:  {only_s3:>5,}  (would be missed without classifier)')
print(f'  All three wrong:      {all_wrong:>5,}')
print()
print(f'Theoretical ceiling (any of the 3 right): {any_right:,} '
      f'({any_right/n*100:.1f}%)')
print('This is the best we could possibly do if we always picked the right method.')
print('The gap between our combined accuracy and this ceiling shows room for improvement')
print('in how we choose between methods.')

Where each step uniquely contributes (validation set):
  All three correct:       83
  Only Step 1 correct:    440  (would be missed without matching)
  Only Step 2 correct:    104  (would be missed without retrieval)
  Only Step 3 correct:    187  (would be missed without classifier)
  All three wrong:      1,376

Theoretical ceiling (any of the 3 right): 1,364 (49.8%)
This is the best we could possibly do if we always picked the right method.
The gap between our combined accuracy and this ceiling shows room for improvement
in how we choose between methods.


## 10. Build the final predictions for the real test set

In [21]:
# Merge Step 2 predictions into the test dataframe
test = test.merge(
    step2[['id', 'step1_code', 'step1_method', 'retrieval_code', 'retrieval_score']],
    on='id',
    how='left'
)

# Apply the same cascade logic
test['final_code'] = test['step1_code']
test['final_method'] = test['step1_method']

# If Step 1 unresolved and retrieval score is high, use retrieval
use_ret = test['final_code'].isna() & (test['retrieval_score'] >= RETRIEVAL_CONFIDENCE_THRESHOLD)
test.loc[use_ret, 'final_code'] = test.loc[use_ret, 'retrieval_code']
test.loc[use_ret, 'final_method'] = 'retrieval'

# Anything still unresolved gets the classifier prediction
use_clf = test['final_code'].isna()
test.loc[use_clf, 'final_code'] = test.loc[use_clf, 'clf_code']
test.loc[use_clf, 'final_method'] = 'classifier'

print('Final method breakdown on the test set:')
print(test['final_method'].value_counts())
print()
print(f'Total predictions: {test["final_code"].notna().sum():,} / {len(test):,}')

Final method breakdown on the test set:
final_method
exact         3812
fuzzy         2819
classifier      24
retrieval       12
Name: count, dtype: int64

Total predictions: 6,667 / 6,667


## 11. Save the outputs

Two files:
- `final_predictions.csv`: everything including the intermediate predictions and confidence scores, useful for analysis
- `kaggle_submission.csv`: clean two-column file ready to upload to Kaggle

In [22]:
detailed_cols = [
    'id', 'Literal', 'Literal_norm',
    'router_pred',
    'step1_code', 'step1_method',
    'retrieval_code', 'retrieval_score',
    'clf_code', 'clf_confidence',
    'final_code', 'final_method'
]
test[detailed_cols].to_csv('final_predictions.csv', index=False)

submission = test[['id', 'final_code']].copy()
submission.columns = ['id', 'Code']
submission.to_csv('kaggle_submission.csv', index=False)

print('Saved:')
print('  final_predictions.csv (full detail with intermediate predictions)')
print('  kaggle_submission.csv (clean file for the Kaggle leaderboard)')
print()
print('Sample of the Kaggle submission:')
submission.head(10)

Saved:
  final_predictions.csv (full detail with intermediate predictions)
  kaggle_submission.csv (clean file for the Kaggle leaderboard)

Sample of the Kaggle submission:


,id,Code
0,1,10903ZU
1,2,E210
2,3,G43909
3,4,B182
4,5,6110
5,6,O3413
6,7,E915
7,8,8500
8,9,Q780
9,10,R011


## 12. Final summary for the presentation

Use these numbers in the final presentation slides. The validation accuracy is our most reliable indicator of true performance because we know the answers there.

In [ ]:
print('PROJECT SUMMARY')
print('=' * 60)
print(f'\nTraining data:  {len(train):,} literals, {train.Code.nunique():,} unique codes')
print(f'Test data:      {len(test):,} literals to predict')
print(f'\nMethod usage on the test set:')
for method, count in test['final_method'].value_counts().items():
    print(f'  {method:<15}: {count:>5,} ({count/len(test)*100:.1f}%)')

print(f'\nValidation accuracy (80/20 split, same seed for all steps):')
print(f'  Step 1 alone (matching):    {val_split.s1_correct.mean()*100:.1f}%')
print(f'  Step 2 alone (retrieval):   {val_split.s2_correct.mean()*100:.1f}%')
print(f'  Step 3 alone (classifier):  {val_split.s3_correct.mean()*100:.1f}%')
print(f'  Combined cascade:           {val_split.combined_correct.mean()*100:.1f}%')

print(f'\nThe combined cascade beats every single step alone,')
print(f'which justifies the multi-strategy pipeline architecture.')

PROJECT SUMMARY

Training data:  13,700 literals, 4,059 unique codes
Test data:      6,667 literals to predict

Method usage on the test set:
  exact          : 3,812 (57.2%)
  fuzzy          : 2,819 (42.3%)
  classifier     :    24 (0.4%)
  retrieval      :    12 (0.2%)

Validation accuracy (80/20 split, same seed for all steps):
  Step 1 alone (matching):    38.0%
  Step 2 alone (retrieval):   10.7%
  Step 3 alone (classifier):  27.2%
  Combined cascade:           38.1%

The combined cascade beats every single step alone,
which justifies the multi-strategy pipeline architecture.


: 